In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.utils.consistency import measure_answer_consistency
from tqdm import tqdm

In [ ]:
df = pd.read_parquet('./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased_categories.parquet')
unique_questions = df.groupby(['title', 'question'])['views'].sum().reset_index()

unique_questions = unique_questions.reset_index(drop=True)
len(unique_questions)

In [ ]:
## Basic stats
print("Min event_year:", df["event_year"].min())
print("Max event_year:", df["event_year"].max())
print(df.count())

In [ ]:
# Filter for years between 2000 and 2026
df = df[(df['event_year'] >= 2000) & (df['event_year'] <= 2026)]

# Display filtered dataframe
df


In [ ]:


# Get the relevant category column name
category_col = "categories"
categories = df[category_col].dropna().unique()

plt.figure(figsize=(12, 7))

# Store color list for up to 4 categories + total
colors = ['#003366', '#990000', '#228B22', '#FF8C00']

# Plot one line for each category (up to 4, else auto-pick colors)
for i, cat in enumerate(sorted(categories)):
    # Aggregate by event_year for this category
    df_cat = df[df[category_col] == cat]
    yearly_cat = (
        df_cat.groupby('event_year')['question']
        .count()
        .reset_index(name='num_questions')
    )
    plt.plot(
        yearly_cat['event_year'],
        yearly_cat['num_questions'],
        marker='o',
        linestyle='-',
        color=colors[i % len(colors)],
        linewidth=2,
        label=str(cat)
    )

# Plot total line (all categories summed)
questions_per_year_total = (
    df.groupby('event_year')['question']
    .count()
    .reset_index(name='num_questions')
)
plt.plot(
    questions_per_year_total['event_year'],
    questions_per_year_total['num_questions'],
    marker='s',
    linestyle='--',
    color='black',
    linewidth=2,
    label='Total'
)

plt.xlabel('Event Year', fontsize=14, fontweight='bold')
plt.ylabel('Number of Temporal Questions', fontsize=14, fontweight='bold')
plt.title('Distribution of Temporal Knowledge Questions by Event Year & Category', fontsize=16, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12, title="Category", title_fontsize=14)
plt.grid(visible=True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:


# Get the relevant category column name
category_col = "categories"
categories = df[category_col].dropna().unique()

plt.figure(figsize=(12, 7))

# Store color list for up to 4 categories + total
colors = ['#003366', '#990000', '#228B22', '#FF8C00']

# Plot one line for each category (up to 4, else auto-pick colors)
for i, cat in enumerate(sorted(categories)):
    # Aggregate sum of views by event_year for this category
    df_cat = df[df[category_col] == cat]
    yearly_cat = (
        df_cat.groupby('event_year')['views']
        .sum()
        .reset_index(name='total_views')
    )
    plt.plot(
        yearly_cat['event_year'],
        yearly_cat['total_views'],
        marker='o',
        linestyle='-',
        color=colors[i % len(colors)],
        linewidth=2,
        label=str(cat)
    )

# Plot total line (all categories summed, views summed per year)
views_per_year_total = (
    df.groupby('event_year')['views']
    .sum()
    .reset_index(name='total_views')
)
plt.plot(
    views_per_year_total['event_year'],
    views_per_year_total['total_views'],
    marker='s',
    linestyle='--',
    color='black',
    linewidth=2,
    label='Total'
)

plt.xlabel('Event Year', fontsize=14, fontweight='bold')
plt.ylabel('Sum of Views', fontsize=14, fontweight='bold')
plt.title('Distribution of Article Views by Event Year & Category', fontsize=16, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12, title="Category", title_fontsize=14)
plt.grid(visible=True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Remove duplicates, keeping first occurrence
df_deduped = df.drop_duplicates(['new_question', 'title', 'event_year'], keep='first')

# Verify deduplication
print(f"Original dataset size: {len(df)}")
print(f"Deduplicated dataset size: {len(df_deduped)}")
print(f"Number of rows removed: {len(df) - len(df_deduped)}")

# Save deduplicated dataset
df_deduped.to_parquet('./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased_categories_deduped.parquet')


In [ ]:
# For each unique question, count the number of unique event_years (i.e., in how many different years does the question appear)
question_year_span = (
    df_deduped
    .groupby('new_question')['event_year']
    .nunique()
    .reset_index(name='num_years')
)

# Plot a histogram: x = how many years a question spans, y = how many questions have that span
plt.figure(figsize=(10, 6))
plt.hist(
    question_year_span['num_years'],
    bins=range(1, 26),
    edgecolor='black',
    alpha=0.75
)
plt.xlabel('Number of Years Question Appears In', fontsize=14, fontweight='bold')
plt.ylabel('Number of Unique Questions', fontsize=14, fontweight='bold')
plt.title('Distribution of Question Lifespan Across Years', fontsize=16, fontweight='bold')
plt.xticks(range(1, 26))
plt.xlim(1, 25)
plt.grid(visible=True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
## Subject check - Answers
df_deduped = pd.read_parquet('./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased_categories_deduped.parquet')
df_subject_not_in_question = df_deduped[
    ~df_deduped.apply(
        lambda row: (
            row['subject'][0].strip().lower() in row['question'].lower()
        ),
        axis=1
    )
]
df_subject_not_in_question

In [ ]:
df_subject_not_in_question['subject'][0][0]

In [ ]:
# Script to measure answer consistency for all questions in df
if 'question' in df.columns and 'answer' in df.columns:
    question_consistency_results = []

    # Use only unique questions
    unique_questions = df['question'].unique()

    for q in tqdm(unique_questions, desc="Measuring answer consistency"):
        score = measure_answer_consistency(q, df, generated_answer_column='answer')
        question_consistency_results.append({'question': q, 'consistency': score})

    df_question_consistency = pd.DataFrame(question_consistency_results)

    # Example: sort by lowest consistency to see most inconsistent questions
    print(df_question_consistency.sort_values('consistency').head(10))
else:
    print("DataFrame does not contain required columns 'question' and 'answer'.")


In [ ]:


plt.figure(figsize=(10, 6))
plt.hist(df_question_consistency['consistency'], bins=20, edgecolor='black')
plt.xlabel('Consistency Score')
plt.ylabel('Number of Questions')
plt.title('Distribution of Question Consistency Scores')
plt.show()